In [5]:
import re
import sys
import json
from pathlib import Path
import pdfplumber

ROW_PATTERN = re.compile(
    r'^(?P<itm>\d+)\s+'
    r'(?P<dos>\d{2}/\d{2}/\d{2})\s+'
    r'(?P<code>D\d{4})\s+'
    r'(?:\S+\s+)*?'
    r'(?P<qty1>\d+)\s+'
    r'(?P<billed>\$[\d,.]+)\s+'
    r'(?P<qty2>\d+)\s+'
    r'(?P<allowed>\$[\d,.]+)\s+'
    r'(?:\d+(?:\.\d+)?%\s+)?'
    r'(?P<money>(?:\$[\d,.]+\s*)+)'
    r'(?P<exc>\S+)\s*$',
    re.MULTILINE
)

TOTALS_PATTERN = re.compile(r'^(?P<money>(?:\$[\d,.]+\s*){9,10})$', re.MULTILINE)

ITEM_NOTE_PATTERN = re.compile(
    r'ITEM:\s*(?P<itm>\d+)\s+'
    r'(?:Service Payment Notes:\s*(?P<notes>.*?)'
    r'(?=\n(?:ITEM:|Authorizations:|Patient Name:|Ref #:|Payee ID:|BILLED ALLOWED|~)|\Z)'
    r'|Exception Code:\s*(?P<exc_code>\S+)\s+(?P<reason>.*?)'
    r'(?=\n(?:ITEM:|Authorizations:|Patient Name:|Ref #:|Payee ID:|BILLED ALLOWED|~)|\Z))',
    re.DOTALL
)

ENCOUNTER_PATTERN = re.compile(r'Encounter\s*#:\s*(\S+)')

ROW_MONEY_FIELDS_8 = [
    "disallow_amount","payable_amount","copay_amount","coins_amount",
    "deduct_amount","patient_pay","other_insur","net_amount"
]
ROW_MONEY_FIELDS_7 = [
    "payable_amount","copay_amount","coins_amount",
    "deduct_amount","patient_pay","other_insur","net_amount"
]
TOTALS_MONEY_FIELDS_10 = [
    "total_disallow_amount","total_payable_amount","total_copay_amount",
    "total_coins_amount","total_deduct_amount","total_patient_pay",
    "total_other_insur","total_net_amount"
]
TOTALS_MONEY_FIELDS_9 = [
    "total_payable_amount","total_copay_amount","total_coins_amount",
    "total_deduct_amount","total_patient_pay","total_other_insur",
    "total_net_amount"
]
MONEY_FIELDS = [
    "billed_amount","allowed_amount","disallow_amount","payable_amount",
    "copay_amount","coins_amount","deduct_amount","patient_pay",
    "other_insur","net_amount"
]

def _is_patient_page(text):
    return "Patient Name:" in text

def _parse_money(value):
    if not value:
        return 0.0
    return float(str(value).replace("$","").replace(",","").strip())

def _fmt_money(value):
    return f"${value:,.2f}"

def _totals_present(totals):
    return bool(totals) and any(v is not None for v in totals.values())

def _resolve_item_note(entries):
    denials = [e for e in entries if e.get("exc_code")]
    if denials:
        codes = ", ".join(e["exc_code"] for e in denials)
        reasons = " | ".join(f'{e["exc_code"]} - {e["reason"]}' for e in denials)
        return {"denial_status":"Denied","denial_reason":reasons}, codes
    return {"denial_status":"Not Denied","denial_reason":None}, None

def _compute_totals_from_services(services):
    totals = {}
    for field in MONEY_FIELDS:
        total = sum(_parse_money(s.get(field)) for s in services)
        totals[f"total_{field}"] = _fmt_money(total)
    return totals

def extract_remittance_pdf(pdf_path):
    pdf_path = Path(pdf_path)
    with pdfplumber.open(pdf_path) as pdf:
        kept_texts = []
        for page in pdf.pages[2:]:
            text = page.extract_text() or ""
            if _is_patient_page(text):
                kept_texts.append(text)
    full_text = "\n".join(kept_texts)
    provider_m = re.search(r'Provider Name:\s*(.+?)\s+Encounter', full_text)
    provider_name = provider_m.group(1).strip() if provider_m else None
    raw_blocks = re.split(r'(?=Patient Name:)', full_text)
    blocks = [b for b in raw_blocks if b.strip().startswith("Patient Name:")]
    patients = [_extract_patient_block(b) for b in blocks]
    patients = _merge_continuation_blocks(patients)
    return {"file_name":pdf_path.name,"provider_name":provider_name,"confidence_score":"100","patients":patients}

def _extract_patient_block(block):
    name_m = re.search(r'Patient Name:\s*(.+?)\s+Provider Name:', block)
    dob_m = re.search(r'DOB:\s*(\d{2}/\d{2}/\d{4})', block)
    enc_m = ENCOUNTER_PATTERN.search(block)
    item_notes = {}

    for m in ITEM_NOTE_PATTERN.finditer(block):
        itm = m.group("itm")
        item_notes.setdefault(itm, [])
        if m.group("exc_code"):
            reason_clean = re.sub(r'\s+',' ',m.group("reason")).strip()
            item_notes[itm].append({"exc_code":m.group("exc_code").strip(),"reason":reason_clean})
        else:
            item_notes[itm].append({"exc_code":None,"reason":None})

    services = []
    for m in ROW_PATTERN.finditer(block):
        itm = m.group("itm")
        note, combined_codes = _resolve_item_note(item_notes.get(itm, []))
        money_values = m.group("money").split()

        if len(money_values) == 8:
            money_fields = dict(zip(ROW_MONEY_FIELDS_8,money_values))
        elif len(money_values) == 7:
            money_fields = dict(zip(ROW_MONEY_FIELDS_7,money_values))
            money_fields["disallow_amount"] = None
        else:
            continue

        service = {
            "item": itm,
            "dos": m.group("dos"),
            "code": m.group("code"),
            "billed_amount": m.group("billed"),
            "allowed_amount": m.group("allowed")
        }
        service.update(money_fields)
        service["exc_code"] = combined_codes if combined_codes else m.group("exc").strip()
        service["denial_status"] = note["denial_status"]
        service["denial_reason"] = note["denial_reason"]
        services.append(service)

    totals_m = TOTALS_PATTERN.search(block)
    if totals_m:
        totals_values = totals_m.group("money").split()
        billed, allowed, rest = totals_values[0], totals_values[1], totals_values[2:]
        if len(rest) == 8:
            totals = dict(zip(TOTALS_MONEY_FIELDS_10,rest))
        elif len(rest) == 7:
            totals = dict(zip(TOTALS_MONEY_FIELDS_9,rest))
            totals["total_disallow_amount"] = None
        else:
            totals = None
        if totals is not None:
            totals["total_billed_amount"] = billed
            totals["total_allowed_amount"] = allowed
    else:
        totals = None

    if not totals:
        totals = {k:None for k in [
            "total_billed_amount","total_allowed_amount","total_disallow_amount",
            "total_payable_amount","total_copay_amount","total_coins_amount",
            "total_deduct_amount","total_patient_pay","total_other_insur","total_net_amount"
        ]}

    return {
        "patient_name":name_m.group(1).strip() if name_m else None,
        "dob":dob_m.group(1) if dob_m else None,
        "_encounter_no":enc_m.group(1).strip() if enc_m else None,
        "_item_notes":item_notes,
        "services":services,
        "totals":totals
    }

def _merge_continuation_blocks(patient_blocks):
    merged = []
    for block in patient_blocks:
        if merged:
            last = merged[-1]
            same_patient = block["patient_name"] == last["patient_name"] and block["dob"] == last["dob"]
            same_encounter = (
                block["_encounter_no"] is not None and
                block["_encounter_no"] == last["_encounter_no"]
            )
            looks_like_continuation = same_patient and (same_encounter or not _totals_present(last["totals"]))

            if looks_like_continuation:
                last["services"].extend(block["services"])
                for itm, entries in block["_item_notes"].items():
                    last["_item_notes"].setdefault(itm,[]).extend(entries)
                if _totals_present(block["totals"]):
                    last["totals"] = block["totals"]
                continue
        merged.append(block)

    for block in merged:
        if not _totals_present(block["totals"]):
            block["totals"] = _compute_totals_from_services(block["services"])

        notes = block.pop("_item_notes")
        for svc in block["services"]:
            note, combined_codes = _resolve_item_note(notes.get(svc["item"],[]))
            if combined_codes:
                svc["exc_code"] = combined_codes
            svc["denial_status"] = note["denial_status"]
            svc["denial_reason"] = note["denial_reason"]

        block.pop("_encounter_no",None)
    return merged

def flatten_for_table(pdf_result):
    rows = []
    for patient in pdf_result["patients"]:
        base = {
            "file_name":pdf_result["file_name"],
            "patient_name":patient["patient_name"],
            "provider_name":pdf_result["provider_name"],
            "dob":patient["dob"],
            **patient["totals"]
        }
        if patient["services"]:
            for svc in patient["services"]:
                row = dict(base)
                row.update(svc)
                rows.append(row)
        else:
            rows.append(base)
    return rows

def run_batch(input_dir,output_path):
    input_dir = Path(input_dir)
    output_path = Path(output_path)
    pdf_files = sorted(input_dir.glob("*.pdf"))

    if not pdf_files:
        print(f"No PDF files found in {input_dir}")
        return

    all_rows = []
    all_results = []
    errors = []

    for pdf_path in pdf_files:
        try:
            result = extract_remittance_pdf(pdf_path)
            all_results.append(result)
            all_rows.extend(flatten_for_table(result))
            n_patients = len(result["patients"])
            n_services = sum(len(p["services"]) for p in result["patients"])
            n_denied = sum(
                1 for p in result["patients"]
                for s in p["services"]
                if s["denial_status"] == "Denied"
            )
            flag = "" if n_patients else "  <-- NO PATIENT BLOCKS FOUND, CHECK THIS FILE"
            print(
                f"OK   {pdf_path.name}: {n_patients} patient(s), "
                f"{n_services} service(s), {n_denied} denied{flag}"
            )
        except Exception as e:
            errors.append((pdf_path.name,str(e)))
            print(f"FAIL {pdf_path.name}: {e}")

    columns = [
        "file_name","patient_name","provider_name","dob","item","dos","code",
        "billed_amount","allowed_amount","disallow_amount","payable_amount",
        "copay_amount","coins_amount","deduct_amount","patient_pay",
        "other_insur","net_amount","exc_code","denial_status","denial_reason",
        "total_billed_amount","total_allowed_amount","total_disallow_amount",
        "total_payable_amount","total_copay_amount","total_coins_amount",
        "total_deduct_amount","total_patient_pay","total_other_insur","total_net_amount"
    ]

    if output_path.suffix.lower() == ".xlsx":
        import openpyxl
        from openpyxl.utils import get_column_letter
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "Remittance"
        ws.append(columns)
        for row in all_rows:
            ws.append([row.get(c,"") for c in columns])
        for i,col in enumerate(columns,start=1):
            max_len = max([len(col)] + [len(str(r.get(col,""))) for r in all_rows])
            ws.column_dimensions[get_column_letter(i)].width = min(max_len+2,40)
        wb.save(output_path)
    else:
        import csv
        with open(output_path,"w",newline="",encoding="utf-8") as f:
            writer = csv.DictWriter(f,fieldnames=columns)
            writer.writeheader()
            for row in all_rows:
                writer.writerow(row)

    json_path = output_path.with_suffix(".json")
    with open(json_path,"w",encoding="utf-8") as f:
        json.dump(all_results,f,indent=2,ensure_ascii=False)

    print(f"\nDone. {len(pdf_files)} PDF(s) processed, {len(errors)} failed.")
    print(f"Table written to:  {output_path}")
    print(f"Raw JSON written to: {json_path}")

    if errors:
        print("\nFiles that failed:")
        for name,err in errors:
            print(f"  - {name}: {err}")

def run_single(pdf_path,output_root="./Test_3"):
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    result = extract_remittance_pdf(pdf_path)
    folder_name = pdf_path.stem
    output_folder = Path(output_root)/folder_name
    output_folder.mkdir(parents=True,exist_ok=True)
    json_path = output_folder/f"{folder_name}.json"

    with open(json_path,"w",encoding="utf-8") as f:
        json.dump([result],f,indent=2,ensure_ascii=False)

    n_patients = len(result["patients"])
    n_services = sum(len(p["services"]) for p in result["patients"])
    n_denied = sum(
        1 for p in result["patients"]
        for s in p["services"]
        if s["denial_status"] == "Denied"
    )

    print("="*70)
    print(f"PDF      : {pdf_path.name}")
    print(f"Patients : {n_patients}")
    print(f"Services : {n_services}")
    print(f"Denied   : {n_denied}")
    print(f"JSON     : {json_path}")
    print("="*70)
    print("\nDENIED SERVICES")
    print("-"*70)

    for patient in result["patients"]:
        for service in patient["services"]:
            if service["denial_status"] == "Denied":
                print(f"Patient : {patient['patient_name']}")
                print(f"Item    : {service['item']}")
                print(f"Code    : {service['code']}")
                print(f"EXC     : {service['exc_code']}")
                print(f"Reason  : {service['denial_reason']}")
                print("-"*70)

    return result,json_path

# single file

In [14]:
pdf_path = r"/home/cipl/users/Yashwanth/Qodoro_OCR/phase_1/EOBs/United_health/UHC_goverment_pdfs/388095333.pdf"
result, json_path = run_single(pdf_path, output_root="./Test_UHC_gov")

PDF      : 388095333.pdf
Patients : 2
Services : 3
Denied   : 1
JSON     : Test_UHC_gov/388095333/388095333.json

DENIED SERVICES
----------------------------------------------------------------------
Patient : ANTONOPOULOS, GEORGE
Item    : 1
Code    : D2740
EXC     : 1098
Reason  : 1098 - Service Authorization Denied
----------------------------------------------------------------------


# For folder

In [13]:
input_dir = Path(r"/home/cipl/users/Jeeva/Phase_2_pdf/ALL_NEW/UHC, UPMC & Geisinger")
output_root = Path("./UMPC_GEI_COM_results_57")

pdf_files = sorted(input_dir.glob("*.pdf"))
total = len(pdf_files)
success = 0
failed = 0

print(f"Total PDF : {total}")
print(f"Success   : {success}")
print(f"Failed    : {failed}")
print(f"Pending   : {total}")

for i, pdf_path in enumerate(pdf_files, 1):
    try:
        run_single(pdf_path, output_root)
        success += 1
    except Exception as e:
        failed += 1
        print(f"FAILED: {pdf_path.name} -> {e}")

    pending = total - success - failed
    print(f"[{i}/{total}] Total: {total} | Success: {success} | Failed: {failed} | Pending: {pending}")

print("\n" + "=" * 50)
print(f"Total PDF : {total}")
print(f"Success   : {success}")
print(f"Failed    : {failed}")
print(f"Pending   : {total - success - failed}")
print("=" * 50)

Total PDF : 57
Success   : 0
Failed    : 0
Pending   : 57
PDF      : $1,140.27.pdf
Patients : 6
Services : 19
Denied   : 1
JSON     : UMPC_GEI_COM_results_57/$1,140.27/$1,140.27.json

DENIED SERVICES
----------------------------------------------------------------------
Patient : BATISTA, RAFAEL
Item    : 1
Code    : D0364
EXC     : 1039
Reason  : 1039 - This service is not covered under the plan.
----------------------------------------------------------------------
[1/57] Total: 57 | Success: 1 | Failed: 0 | Pending: 56
PDF      : $1,156.12.pdf
Patients : 4
Services : 18
Denied   : 0
JSON     : UMPC_GEI_COM_results_57/$1,156.12/$1,156.12.json

DENIED SERVICES
----------------------------------------------------------------------
[2/57] Total: 57 | Success: 2 | Failed: 0 | Pending: 55
PDF      : $1,174.95.pdf
Patients : 8
Services : 56
Denied   : 4
JSON     : UMPC_GEI_COM_results_57/$1,174.95/$1,174.95.json

DENIED SERVICES
-------------------------------------------------------------

In [12]:
input_dir = Path(r"/home/cipl/users/Jeeva/Phase_2_pdf/ALL_NEW/UHC, UPMC & Geisinger")

pdf_files = sorted(input_dir.glob("*.pdf"))
total = len(pdf_files)
success = 0
failed = 0

print(f"Total PDF : {total}")
print(f"Success   : {success}")
print(f"Failed    : {failed}")
print(f"Pending   : {total}")
print("-" * 50)

for i, pdf_path in enumerate(pdf_files, 1):
    try:
        result = extract_remittance_pdf(pdf_path)
        success += 1
        print(f"[{i}/{total}] SUCCESS : {pdf_path.name}")
    except Exception as e:
        failed += 1
        print(f"[{i}/{total}] FAILED  : {pdf_path.name} -> {e}")

    pending = total - success - failed
    print(f"Total: {total} | Success: {success} | Failed: {failed} | Pending: {pending}")

print("\n" + "=" * 50)
print("FINAL COUNT")
print("=" * 50)
print(f"Total PDF : {total}")
print(f"Success   : {success}")
print(f"Failed    : {failed}")
print(f"Pending   : {total - success - failed}")
print("=" * 50)

Total PDF : 57
Success   : 0
Failed    : 0
Pending   : 57
--------------------------------------------------
[1/57] SUCCESS : $1,140.27.pdf
Total: 57 | Success: 1 | Failed: 0 | Pending: 56
[2/57] SUCCESS : $1,156.12.pdf
Total: 57 | Success: 2 | Failed: 0 | Pending: 55
[3/57] SUCCESS : $1,174.95.pdf
Total: 57 | Success: 3 | Failed: 0 | Pending: 54
[4/57] SUCCESS : $1,431.77.pdf
Total: 57 | Success: 4 | Failed: 0 | Pending: 53
[5/57] SUCCESS : $1,550.00.pdf
Total: 57 | Success: 5 | Failed: 0 | Pending: 52
[6/57] SUCCESS : $1,580.00.pdf
Total: 57 | Success: 6 | Failed: 0 | Pending: 51
[7/57] SUCCESS : $1,665.00.pdf
Total: 57 | Success: 7 | Failed: 0 | Pending: 50
[8/57] SUCCESS : $1,699.00.pdf
Total: 57 | Success: 8 | Failed: 0 | Pending: 49
[9/57] SUCCESS : $1,748.12.pdf
Total: 57 | Success: 9 | Failed: 0 | Pending: 48
[10/57] SUCCESS : $1,985.50.pdf
Total: 57 | Success: 10 | Failed: 0 | Pending: 47
[11/57] SUCCESS : $11,228.00.pdf
Total: 57 | Success: 11 | Failed: 0 | Pending: 46
[12/57